In [1]:
!pip install langchain langchain_core langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.6/987.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.7/371.7 kB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.5/129.5 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 18.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.8 MB/s eta 0:00:00


In [2]:
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field, validator
from langchain_openai import ChatOpenAI

from datetime import datetime

from typing import List, Tuple

In [3]:
import os
import getpass

os.environ['OPENAI_API_KEY'] = getpass.getpass("OpenAI API Key: ")

OpenAI API Key: ··········


In [4]:
wired_brain_items = set(["Latte", "Espresso", "Capuccino", "Croissant", "Bagel", "Hot Chocolate"])

In [5]:
tx_log = """
Date: January 15, 2022, 8:30 AM
Customer: Joe Smith
1 x Latte      3.50
2 x Croissant 2.00
1 x Espresso  2.50
Total: 10.00

Date: January 15, 2022, 8:40 AM
Customer: Leroy Thompson
1 x Hot Chocolate  3.25
1 x Bagel          2.00
Total: 5.25

Date: January 15, 2022, 9:15 AM
Customer: Jane Doe
2 x Capuccino 3.75
2 x Croissant 2.00
Total: 11.50
"""

In [6]:
class SalesReceipt(BaseModel):
  timestamp: datetime = Field(description="date and time of transaction")
  items: List[Tuple[str, float, int]] = Field(description="list of items purchased, the cost of the item, and the quantity purchased")
  total: float = Field(description="total cost of all items purchased")

  @validator('timestamp')
  def validate_timestamp(cls, v):
    if v < datetime(2008, 3, 15):
      raise ValueError("Transaction date must be after 3/15/2008")
    elif v > datetime.now():
      raise ValueError("Transaction date must be before today")
    return v
  @validator('items')
  def validate_items(cls, v):
    for x in v:
      if x[0] not in wired_brain_items:
        raise ValueError(f"{x[0]} is not a valid item")
    return v

class ReceiptList(BaseModel):
  receipts: List[SalesReceipt]

In [7]:
model = ChatOpenAI(temperature=0)

In [8]:
parser = PydanticOutputParser(pydantic_object=ReceiptList)

prompt = PromptTemplate(
    template = "Generate receipts based on the following transaction log.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | model | parser

In [9]:
chain.invoke({"query": tx_log})

ReceiptList(receipts=[SalesReceipt(timestamp=datetime.datetime(2022, 1, 15, 8, 30), items=[('Latte', 3.5, 1), ('Croissant', 2.0, 2), ('Espresso', 2.5, 1)], total=10.0), SalesReceipt(timestamp=datetime.datetime(2022, 1, 15, 8, 40), items=[('Hot Chocolate', 3.25, 1), ('Bagel', 2.0, 1)], total=5.25), SalesReceipt(timestamp=datetime.datetime(2022, 1, 15, 9, 15), items=[('Capuccino', 3.75, 2), ('Croissant', 2.0, 2)], total=11.5)])

In [10]:
bad_tx_log = """
Date: December 19, 2007, 8:30 AM
Customer: Joe Smith
1 x Ice Cream 5.50
2 x Croissant 2.00
1 x Espresso  2.50
Total: 12.00

Date: January 15, 2022, 8:40 AM
Customer: Leroy Thompson
1 x Hot Chocolate  3.25
1 x Bagel          2.00
Total: 5.25
"""

In [11]:
chain.invoke({"query": bad_tx_log})

OutputParserException: Failed to parse ReceiptList from completion {"receipts": [{"timestamp": "2007-12-19T08:30:00", "items": [["Ice Cream", 5.5, 1], ["Croissant", 2.0, 2], ["Espresso", 2.5, 1]], "total": 12.0}, {"timestamp": "2022-01-15T08:40:00", "items": [["Hot Chocolate", 3.25, 1], ["Bagel", 2.0, 1]], "total": 5.25}]}. Got: 2 validation errors for ReceiptList
receipts -> 0 -> timestamp
  Transaction date must be after 3/15/2008 (type=value_error)
receipts -> 0 -> items
  Ice Cream is not a valid item (type=value_error)